# Orchestration & Durability

An end-to-end tour of grasp-agents' composition and persistence machinery, on real LLMs:

1. **Composable architectures** — LLM agents, plain processors, parallel fan-out, and sequential/looped workflows snapped together
2. **Agent team** — a `Runner` graph with dynamic, content-based routing
3. **Checkpointing** — crash & resume for agents, workflows, runner teams, and background tasks on a `FileCheckpointStore` (a Bash task is reported on resume; a sub-agent is re-spawned and continues)
4. **Sandboxed data processing** — Bash + a persistent Python kernel under `srt` confinement with a network allowlist

Needs `OPENAI_API_KEY` in `.env` (everything runs on `gpt-5.4-nano`). Sections 3.4 and 4 additionally need the `srt` CLI (`npm install -g @anthropic-ai/sandbox-runtime`) and the `notebook` extra (`pip install grasp-agents[notebook]`).

---
## Setup

In [ ]:
import asyncio
import shutil
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from grasp_agents import (
    LLMAgent,
    LoopedWorkflow,
    ParallelProcessor,
    Processor,
    ProcPacketOutEvent,
    SessionContext,
    Runner,
    RunPacketOutEvent,
    SequentialWorkflow,
    function_tool,
    render_events,
)
from grasp_agents.durability import FileCheckpointStore
from grasp_agents.llm_providers.openai_responses import OpenAIResponsesLLM
from grasp_agents.types.packet import Packet
from grasp_agents.runner.runner import END_PROC_NAME

load_dotenv()

WORKDIR = Path("./orchestration_workdir").resolve()
WORKDIR.mkdir(exist_ok=True)


def make_llm(
    max_output_tokens: int = 800, *, structured: bool = False
) -> OpenAIResponsesLLM:
    return OpenAIResponsesLLM(
        model_name="gpt-5.4-nano",
        llm_settings={"max_output_tokens": max_output_tokens},
        apply_output_schema_via_provider=structured,
    )


async def run_streamed(proc, *args, show_background=False, **kwargs):
    """
    Run any processor and watch it happen: render_events() prints every turn,
    tool call, and routing decision as it streams, instead of blocking on an
    opaque `await proc.run(...)`. (`.run()` is just `.run_stream()` drained for
    you.) `show_background=True` also prints background subagents' / tasks'
    internal events (off by default). Returns the final payloads.
    """
    final = None
    async for event in render_events(
        proc.run_stream(*args, **kwargs), show_background=show_background
    ):
        # The run's final output: a Runner emits RunPacketOutEvent; every other
        # processor emits a ProcPacketOutEvent tagged with its own name.
        is_final = isinstance(event, RunPacketOutEvent) or (
            isinstance(event, ProcPacketOutEvent) and event.source == proc.name
        )
        if is_final:
            final = event.data.payloads
    return final


async def simulate_process_death(agent):
    """
    Fake a hard crash in this single notebook process: cancel the agent's
    in-flight background tasks so they vanish as they would if the process were
    killed — but WITHOUT finalizing their records. (`agent.aclose()` would mark
    the records cancelled, and cancelled records are skipped on resume.) The
    checkpoints + task records persist on disk; that's what resume works from.
    """
    tasks = [bt.consumer for bt in agent.agent_ctx.bg_tasks._tasks.values()]  # noqa: SLF001
    for t in tasks:
        t.cancel()
    await asyncio.gather(*tasks, return_exceptions=True)

---
## 1. Composable architectures

Everything is a `Processor[InT, OutT, CtxT]` — LLM agents, plain Python steps,
parallel fan-outs, even whole workflows — so they compose freely. We build a
report pipeline that mixes all of them:

```
"brief" ──► report_pipeline (SequentialWorkflow)
              ├─ planner     LLMAgent[str, ReportPlan]            structured output
              ├─ splitter    Processor[ReportPlan, SectionSpec]   1 payload → N payloads
              ├─ writers     ParallelProcessor(section_writer)    one replica per section
              └─ assembler   Processor[DraftSection, str]         join into one document
```

The packet flowing between steps can carry many payloads — `ParallelProcessor`
runs one replica of its sub-processor per payload, concurrently.

In [ ]:
class SectionSpec(BaseModel):
    title: str
    angle: str = Field(description="The specific question this section answers")


class ReportPlan(BaseModel):
    sections: list[SectionSpec]


class DraftSection(BaseModel):
    title: str
    text: str


class Splitter(Processor[ReportPlan, SectionSpec, None]):
    """One ReportPlan payload in -> one payload per section out."""

    async def _process(
        self, chat_inputs=None, *, in_args=None, exec_id, step=None
    ) -> list[SectionSpec]:
        del chat_inputs, exec_id, step
        return [s for plan in in_args or [] for s in plan.sections]


class Assembler(Processor[DraftSection, str, None]):
    """Join the drafted sections into one markdown report."""

    async def _process(
        self, chat_inputs=None, *, in_args=None, exec_id, step=None
    ) -> list[str]:
        del chat_inputs, exec_id, step
        doc = "\n\n".join(f"### {d.title}\n{d.text}" for d in in_args or [])
        return [doc]

In [ ]:
planner = LLMAgent[str, ReportPlan, None](
    name="planner",
    llm=make_llm(structured=True),
    llm_output_schema=ReportPlan,
    sys_prompt=(
        "Plan a short report on the given brief: exactly 3 sections, each "
        "with a sharp title and the specific question it answers."
    ),
    max_retries=2,
)

section_writer = LLMAgent[SectionSpec, DraftSection, None](
    name="section_writer",
    llm=make_llm(structured=True),
    llm_output_schema=DraftSection,
    in_prompt=(
        "Write the report section '{title}' (2-3 sentences) answering: {angle}"
    ),
    sys_prompt="You write tight, factual report sections.",
    max_retries=2,
)

report_pipeline = SequentialWorkflow[str, str, None](
    name="report_pipeline",
    subprocs=[
        planner,
        Splitter(name="splitter"),
        ParallelProcessor[SectionSpec, DraftSection, None](subproc=section_writer),
        Assembler(name="assembler"),
    ],
)

final = await run_streamed(
    report_pipeline, "Why small specialized LLM agents beat one giant prompt"
)
print("\n=== assembled report ===\n", final[0])

### Looped workflow

`LoopedWorkflow` cycles its sub-processors until the exit processor's output
satisfies a terminator (or `max_iterations` is hit). Here a *polisher* rewrites
an abstract until a *critic* approves it:

```
rough ──► [ polisher ──► critic ] ──► APPROVED? ── yes ─► out
              ▲                          │ no
              └──────── feedback ────────┘
```

In [ ]:
polisher = LLMAgent[str, str, None](
    name="polisher",
    llm=make_llm(),
    sys_prompt=(
        "Polish the given one-paragraph abstract: concrete, plain language, "
        "no buzzwords. If you received reviewer feedback (REVISE: ...), apply "
        "it to your previous draft. Return only the improved paragraph."
    ),
)

critic = LLMAgent[str, str, None](
    name="critic",
    llm=make_llm(),
    sys_prompt=(
        "You review a one-paragraph abstract. If it is clear, specific, and "
        "buzzword-free, reply 'APPROVED: <the paragraph>'. Otherwise reply "
        "'REVISE: <one concrete instruction>'."
    ),
)

polish_loop = LoopedWorkflow[str, str, None](
    name="polish_loop",
    subprocs=[polisher, critic],
    exit_proc=critic,
    max_iterations=3,
)


@polish_loop.add_workflow_loop_terminator
def approved(out_packet: Packet[str], **kwargs: Any) -> bool:
    return out_packet.payloads[0].strip().startswith("APPROVED")


rough = (
    "Our system leverages synergistic multi-agent paradigms to operationalize "
    "holistic, next-generation LLM-driven value across the enterprise."
)
final = await run_streamed(polish_loop, in_args=[rough])
print("\n=== final abstract ===\n", final[0])

---
## 2. Agent team (`Runner`)

A `Runner` executes a *graph* of processors connected by `recipients`. Routing
is dynamic: any processor can pick its recipients per payload with
`add_recipient_selector` — based on content, state, or LLM judgment.

```
ticket ──► triage ──┬──► billing ──┐
                    └──► tech ─────┴──► reviewer ──► END
```

In [ ]:
ctx_team = SessionContext[None](state=None)

triage = LLMAgent[str, str, None](
    name="triage",
    llm=make_llm(),
    recipients=["billing", "tech"],
    sys_prompt=(
        "Classify the support ticket. Reply with exactly one line: "
        "'BILLING: <one-line summary>' or 'TECH: <one-line summary>'."
    ),
    reset_transcript_on_run=True,
)


@triage.add_recipient_selector
def route_ticket(output: str, **kwargs: Any) -> list[str]:
    is_billing = output.strip().upper().startswith("BILLING")
    return ["billing" if is_billing else "tech"]


billing = LLMAgent[str, str, None](
    name="billing",
    llm=make_llm(),
    recipients=["reviewer"],
    sys_prompt=(
        "You are a billing support specialist. Resolve the summarized ticket "
        "in 2-4 sentences with a concrete next step."
    ),
    reset_transcript_on_run=True,
)

tech = LLMAgent[str, str, None](
    name="tech",
    llm=make_llm(),
    recipients=["reviewer"],
    sys_prompt=(
        "You are a technical support specialist. Resolve the summarized ticket "
        "in 2-4 sentences with a concrete next step."
    ),
    reset_transcript_on_run=True,
)

reviewer = LLMAgent[str, str, None](
    name="reviewer",
    llm=make_llm(),
    recipients=[END_PROC_NAME],
    sys_prompt=(
        "Polish the specialist's reply into the final customer message: "
        "friendly, concise, ending with one clear next step. Sign as "
        "'— SupportBot'."
    ),
    reset_transcript_on_run=True,
)

support_desk = Runner[str, None](
    entry_proc=triage,
    procs=[triage, billing, tech, reviewer],
    ctx=ctx_team,
    name="support_desk",
)

In [ ]:
ticket = (
    "I was double-charged after upgrading my plan, and the invoice email "
    "links to a 404 page."
)

final = await run_streamed(support_desk, ticket)
print("\n=== customer reply (via billing) ===\n", final[0])

In [ ]:
# Same team, different content -> the triage agent routes it to `tech` instead.
final = await run_streamed(
    support_desk,
    "The dashboard shows a blank page after login; the browser console says "
    "'TypeError: t is undefined'.",
)
print("\n=== customer reply (via tech) ===\n", final[0])

---
## 3. Checkpointing & resume

Processors bound to a context with a `checkpoint_store` persist their progress
as they run — each kind saves what it needs to continue, keyed by
`<session_key>/<kind>/<path>`:

| kind | what is persisted |
|---|---|
| agent | transcript (append-only message log), turn, pending background tasks |
| workflow | last completed step + its output packet |
| runner | in-flight routed events between processors |
| task | one durable record per background task |

The recovery recipe is always the same: **rebuild the processors, bind them to
the same `session_key` + store, and run** — completed work is restored, not
re-executed. An `LLMAgent` run with *no inputs at all* is a *pure resume*.

We simulate crashes with hooks/flaky steps that raise mid-run.

In [ ]:
import contextlib
import logging

store_root = WORKDIR / "checkpoints"
shutil.rmtree(store_root, ignore_errors=True)  # a fresh store for the demo
store = FileCheckpointStore(store_root)


@contextlib.contextmanager
def expected_crash():
    """
    Silence the framework's ERROR-level traceback for a crash we trigger on
    purpose (e.g. the Runner's EventBus logs an unexpected handler failure).
    Use ONLY around deliberate demo crashes — never hide real errors.
    """
    lg = logging.getLogger("grasp_agents")
    prev = lg.level
    lg.setLevel(logging.CRITICAL)
    try:
        yield
    finally:
        lg.setLevel(prev)

### 3.1 Agent: crash mid-tool-loop, pure resume

The agent needs two tool calls. We let the first one complete, then "crash the
process" right before the second LLM turn. On resume the agent continues from
its checkpointed transcript — the first addition is **not** re-done.

In [ ]:
calls = {"add": 0}


@function_tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    calls["add"] += 1
    return a + b


CALC_PROMPT = (
    "You are a calculator. Use the add tool for EVERY addition, one call per "
    "turn. Return just the final number."
)

ctx_a1 = SessionContext[None](
    state=None, checkpoint_store=store, session_key="solo-agent"
)
crashy = LLMAgent[str, str, None](
    name="calculator", ctx=ctx_a1, llm=make_llm(), tools=[add],
    sys_prompt=CALC_PROMPT,
)

hook_hits = {"n": 0}


@crashy.add_before_llm_hook
async def crash_on_second_llm_call(**kwargs: Any) -> None:
    hook_hits["n"] += 1
    if hook_hits["n"] == 2:
        raise RuntimeError("simulated process crash")


try:
    await run_streamed(
        crashy, "Compute (17.5 + 24.5), then add 100 to the result."
    )
except Exception as exc:
    print(f"\ncrashed as planned: {exc!r}")
print("add-tool calls before the crash:", calls["add"])

In [ ]:
# "New process": fresh agent + fresh context, same session_key + store.
calls["add"] = 0
ctx_a2 = SessionContext[None](
    state=None, checkpoint_store=store, session_key="solo-agent"
)
resumed = LLMAgent[str, str, None](
    name="calculator", ctx=ctx_a2, llm=make_llm(), tools=[add],
    sys_prompt=CALC_PROMPT,
)

# no inputs -> pure resume; watch it pick up mid-loop from the checkpoint
final = await run_streamed(resumed)
print("\nfinal answer:", final[0])
print(
    "add-tool calls on resume:", calls["add"],
    "(first addition restored, not re-run)",
)

### 3.2 Workflow: a completed step is never re-run

The drafter (an LLM call) completes; the formatter crashes. On resume the
workflow restarts **at the failed step**, feeding it the checkpointed packet —
watch the drafter's LLM call count stay at 1.

In [ ]:
llm_calls = {"drafter": 0}
flaky = {"failed_once": False}


class FlakyFormatter(Processor[str, str, None]):
    """Pretends the downstream formatting service is down on its first call."""

    async def _process(
        self, chat_inputs=None, *, in_args=None, exec_id, step=None
    ) -> list[str]:
        del chat_inputs, exec_id, step
        if not flaky["failed_once"]:
            flaky["failed_once"] = True
            raise RuntimeError("formatting service unavailable")
        return ["*** " + (in_args or [""])[0].strip() + " ***"]


def make_drafter() -> LLMAgent[str, str, None]:
    drafter = LLMAgent[str, str, None](
        name="drafter",
        llm=make_llm(),
        sys_prompt="Write exactly one vivid sentence about the given topic.",
    )

    @drafter.add_after_llm_hook
    async def count_llm_calls(**kwargs: Any) -> None:
        llm_calls["drafter"] += 1

    return drafter


ctx_w1 = SessionContext[None](
    state=None, checkpoint_store=store, session_key="workflow-demo"
)
wf1 = SequentialWorkflow[str, str, None](
    name="draft_format",
    subprocs=[make_drafter(), FlakyFormatter(name="formatter")],
    ctx=ctx_w1,
)

try:
    await run_streamed(wf1, "a lighthouse in a winter storm")
except Exception as exc:
    print(f"\ncrashed as planned: {exc!r}")
print("drafter LLM calls:", llm_calls["drafter"])

In [ ]:
ctx_w2 = SessionContext[None](
    state=None, checkpoint_store=store, session_key="workflow-demo"
)
wf2 = SequentialWorkflow[str, str, None](
    name="draft_format",
    subprocs=[make_drafter(), FlakyFormatter(name="formatter")],
    ctx=ctx_w2,
)

final = await run_streamed(wf2, "a lighthouse in a winter storm")
print("\nfinal output:", final[0])
print(
    "drafter LLM calls after resume:", llm_calls["drafter"],
    "(step 0 restored from the checkpoint)",
)

### 3.3 Runner team: pending work survives the crash

Inside a `Runner`, each processor checkpoints itself **and** the runner
checkpoints the events in flight between them. The picker (an LLM agent)
finishes, the downstream publisher crashes — on resume only the publisher
runs.

In [ ]:
llm_calls["picker"] = 0
flaky["publish_failed_once"] = False


class FlakyPublisher(Processor[str, str, None]):
    async def _process(
        self, chat_inputs=None, *, in_args=None, exec_id, step=None
    ) -> list[str]:
        del chat_inputs, exec_id, step
        if not flaky["publish_failed_once"]:
            flaky["publish_failed_once"] = True
            raise RuntimeError("publishing API timeout")
        return ["PUBLISHED: " + (in_args or [""])[0]]


def make_picker() -> LLMAgent[str, str, None]:
    picker = LLMAgent[str, str, None](
        name="picker",
        llm=make_llm(),
        recipients=["publisher"],
        sys_prompt=(
            "Pick ONE punchy headline for the given topic. Reply with just "
            "the headline."
        ),
    )

    @picker.add_after_llm_hook
    async def count_picker_calls(**kwargs: Any) -> None:
        llm_calls["picker"] += 1

    return picker


def build_news_team(ctx: SessionContext[None]) -> Runner[str, None]:
    picker = make_picker()
    publisher = FlakyPublisher(name="publisher", recipients=[END_PROC_NAME])
    return Runner[str, None](
        entry_proc=picker, procs=[picker, publisher], ctx=ctx, name="newsdesk"
    )


ctx_r1 = SessionContext[None](
    state=None, checkpoint_store=store, session_key="team-demo"
)
# expected_crash() hushes the EventBus's ERROR log for this on-purpose failure.
with expected_crash():
    try:
        await run_streamed(
            build_news_team(ctx_r1), "city greenlights rooftop solar on schools"
        )
    except Exception as exc:
        print(f"\ncrashed as planned: {type(exc).__name__}")
print("picker LLM calls:", llm_calls["picker"])

In [ ]:
ctx_r2 = SessionContext[None](
    state=None, checkpoint_store=store, session_key="team-demo"
)
# no inputs: the checkpointed pending events drive the resumed team
final = await run_streamed(build_news_team(ctx_r2))
print("\nfinal output:", final[0])
print(
    "picker LLM calls after resume:", llm_calls["picker"],
    "(the agent was not re-invoked)",
)

### 3.4 Resuming a batch job from its saved log (Bash)

A slow Bash command that outlives `auto_background_at` moves to the background.
The command itself is **not** resurrected on resume — but its streamed output is
durably mirrored to `.grasp/tasks/<id>.log`. So when the process dies mid-job,
the resumed agent is told the task was interrupted (with the log's path) and
**picks up where it left off**: it reads the log to see which items already
finished, then processes only the ones still missing — no redone work.

*(Contrast §3.5: a backgrounded **sub-agent** IS re-spawned from its checkpoint
and continues automatically.)*

We pass `show_background=True` so the backgrounded command's live output prints
inline (it's suppressed by default). Needs the `srt` CLI.

In [ ]:
from grasp_agents.sandbox import NetworkPolicy, local_environment
from grasp_agents.tools.bash import bash_tools

BG_DIR = WORKDIR / "bg_workspace"
BG_DIR.mkdir(exist_ok=True)

BG_PROMPT = (
    "You run a slow batch job to completion. A shell command processes the batch's "
    "items one at a time. The user will provide the exact command to run. You "
    "need to make sure the whole batch gets processed. Avoid processing the same item twice."
)


def build_bg_agent(session_ctx: SessionContext[None]) -> LLMAgent[str, str, None]:
    return LLMAgent[str, str, None](
        name="bg_agent",
        ctx=session_ctx,
        llm=make_llm(),
        # block_leading_sleep is an anti-spin guard (it stops the model from
        # `sleep`-ing to stall the loop); off here because this demo's whole
        # point is to launch a deliberately slow command and background it.
        tools=[*bash_tools(auto_background_at=0.5, block_leading_sleep=False)],
        sys_prompt=BG_PROMPT,
    )


ctx_b1 = SessionContext[None](
    state=None,
    checkpoint_store=store,
    session_key="bg-demo",
    environment=local_environment(
        allowed_roots=[BG_DIR], confinement="srt", network=NetworkPolicy.NONE
    ),
)
bg_agent = build_bg_agent(ctx_b1)

bg_hook_hits = {"n": 0}


@bg_agent.add_before_llm_hook
async def crash_while_task_pending(**kwargs: Any) -> None:
    bg_hook_hits["n"] += 1
    if bg_hook_hits["n"] == 2:  # the LLM call right after the task launched
        raise RuntimeError("simulated crash while the background task runs")


try:
    # show_background=True → the backgrounded command's live output prints here.
    await run_streamed(
        bg_agent,
        "Process this batch of 5 items by running exactly this command: "
        'for i in 1 2 3 4 5; do echo "item $i of 5 done"; sleep 2; done',
        show_background=True,
    )
except Exception as exc:
    print(f"\ncrashed as planned: {exc!r}")

await simulate_process_death(bg_agent)  # leaves the Bash record PENDING

In [ ]:
# Resume in a "new process". The Bash task is NOT re-run for us; instead the
# agent is told it was interrupted (with the saved-log path), reads it to see
# which items already finished, and processes only the ones still missing.

# On resume the interruption notice streams as a normal event (you'll see it
# rendered above); the agent reads the saved log and finishes the remaining
# items only.

ctx_b2 = SessionContext[None](
    state=None,
    checkpoint_store=store,
    session_key="bg-demo",
    environment=local_environment(
        allowed_roots=[BG_DIR], confinement="srt", network=NetworkPolicy.NONE
    ),
)
resumed_bg = build_bg_agent(ctx_b2)

final = await run_streamed(resumed_bg, show_background=True)  # pure resume

# The durable artifact the agent read: the task's streamed output, captured
# before the crash.
log = next((BG_DIR / ".grasp" / "tasks").glob("*.log"))
print(f"\n=== saved task log ({log.name}) ===\n{log.read_text().strip()}")

### 3.5 Resumable background tasks: sub-agents that survive a crash

The flip side of §3.4. A backgrounded **sub-agent** *is* resumable: its own
turns are checkpointed, so on resume the framework **re-spawns it from its
checkpoint and it continues where it left off** — nothing redone, no manual
"redo it".

A `coordinator` launches two researchers in the background; each does a slow
lookup, so they're mid-investigation when we crash. On resume both are
re-spawned — with `show_background=True` you watch them pick up where they left
off and finish, and the coordinator delivers a briefing citing both.

In [ ]:
from grasp_agents.tools.agent_tool import AgentTool


@function_tool
async def slow_lookup(topic: str) -> str:
    """A slow reference lookup (a few seconds)."""
    await asyncio.sleep(5)
    return f"reference material on {topic}"


def researcher(name: str, topic: str) -> AgentTool[None]:
    return AgentTool[None](
        name=name,
        description=f"Research {topic}; returns one vivid fact.",
        llm=make_llm(),
        tools=[slow_lookup],
        sys_prompt=(
            f"Research '{topic}'. First call slow_lookup with the topic, then "
            f"reply with ONE vivid one-sentence fact mentioning {topic}."
        ),
        auto_background_at=0,  # background immediately
        max_turns=4,
    )


def build_coordinator(ctx: SessionContext[None]) -> LLMAgent[str, str, None]:
    return LLMAgent[str, str, None](
        name="coordinator",
        ctx=ctx,
        llm=make_llm(),
        tools=[
            researcher("reef_researcher", "coral reefs"),
            researcher("volcano_researcher", "volcanoes"),
        ],
        sys_prompt=(
            "In your FIRST turn, call BOTH reef_researcher AND "
            "volcano_researcher (they run in the background). Then wait for "
            "both and reply with a two-line briefing citing each finding."
        ),
        max_turns=8,
    )


ctx_s1 = SessionContext[None](
    state=None, checkpoint_store=store, session_key="team-research"
)
coordinator = build_coordinator(ctx_s1)

sub_hits = {"n": 0}


@coordinator.add_before_llm_hook
async def crash_mid_research(**kwargs: Any) -> None:
    sub_hits["n"] += 1
    if sub_hits["n"] == 2:  # turn 1 launched both researchers; crash now
        raise RuntimeError("simulated crash mid-research")


try:
    await run_streamed(
        coordinator, "Brief me on coral reefs and volcanoes.", show_background=True
    )
except Exception as exc:
    print(f"\ncrashed as planned: {exc!r}")

# Interrupt the in-flight researchers, leaving their records PENDING.
await simulate_process_death(coordinator)

In [ ]:
ctx_s2 = SessionContext[None](
    state=None, checkpoint_store=store, session_key="team-research"
)
resumed_coord = build_coordinator(ctx_s2)

# show_background=True → watch the re-spawned researchers pick up where they
# left off (their turns stream in) and finish; then the coordinator briefs.
async with resumed_coord:
    final = await run_streamed(resumed_coord, show_background=True)  # pure resume
print("\n=== coordinator briefing after resume ===\n", final[0])

### 3.6 What's on disk

The store lays sessions out as `<session_key>/<kind>/<path>` — every demo
above left its trace:

In [ ]:
for path in sorted(store_root.rglob("*")):
    if path.is_file():
        print(path.relative_to(store_root))

---
## 4. Sandboxed data processing (Bash + Python under srt)

File, terminal, and code tools run under a two-plane `SandboxPolicy`: a file
plane (writes confined to `allowed_roots`) and an OS plane (process
confinement + a **network allowlist**). Here the analyst may reach
`raw.githubusercontent.com` — and nothing else.

`provision=True` gives the agent its OWN dedicated venv at `<workspace>/.venv`
(created from `sys.executable`, with the listed `packages`) — so the demo is
self-contained and the agent's installs never touch your environment. `python`
could pin a different base/version. `RunPython` then runs a persistent Jupyter
kernel on that venv: variables survive across calls, and figures the code
displays come back to the model (and render below) as images.

*(The first call builds the venv + `pip install`s — that one-time setup runs
on the host, before confinement, so it isn't subject to the allowlist.)*

In [ ]:
from grasp_agents.tools.code_interpreter import RunPython

DATA_DIR = WORKDIR / "data_workspace"
DATA_DIR.mkdir(exist_ok=True)

env = local_environment(
    allowed_roots=[DATA_DIR],
    confinement="srt",
    network=NetworkPolicy.ALLOWLIST,
    allowed_domains=["raw.githubusercontent.com"],
    provision=True,  # dedicated agent venv at DATA_DIR/.venv
    packages=["ipykernel", "numpy", "pandas", "matplotlib"],
    env={"MPLCONFIGDIR": str(DATA_DIR / ".mpl")},
    kernel_setup_code="%matplotlib inline",
)
ctx_data = SessionContext[None](state=None, environment=env)

analyst = LLMAgent[str, str, None](
    name="analyst",
    ctx=ctx_data,
    llm=make_llm(2000),
    tools=[*bash_tools(auto_background_at=120), RunPython()],
    sys_prompt=(
        "You are a data analyst. Bash and RunPython (a persistent Python "
        "kernel with numpy, pandas and matplotlib) share one workspace — "
        "your cwd. Work step by step, one tool call at a time. Show charts "
        "by drawing with matplotlib and calling plt.show()."
    ),
    max_turns=16,
    stream_llm=True,
)

In [ ]:
TASK = (
    "1) Bash: download "
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv "
    "to penguins.csv with `curl -fsSL` (quiet, fail on error), then show its "
    "line count and first 3 lines.\n"
    "2) Bash: count rows per species with awk (skip the header).\n"
    "3) RunPython: load penguins.csv with pandas, compute the mean "
    "body_mass_g per species (skip missing values), draw a labeled bar chart, "
    "save it to penguins_mass.png with fig.savefig(...), THEN plt.show() it "
    "(save before show — the inline backend closes the figure on show).\n"
    "4) Bash: try downloading https://example.com/data.csv and report in one "
    "line what happens (the sandbox should refuse it).\n"
    "Finish with a 3-line summary of the findings."
)

await run_streamed(analyst, TASK)

In [ ]:
print(sorted(p.name for p in DATA_DIR.iterdir() if not p.name.startswith(".")))

# The agent's session resources (the RunPython kernel, shells) live until the
# agent is closed:
await analyst.aclose()